In [1]:
SEED=15179996

!pip install -q -U transformers peft bitsandbytes datasets optuna evaluate scikit-learn accelerate tqdm pandas huggingface_hub torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 142.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 154.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cu

In [ ]:
import torch
import pandas as pd
import numpy as np
import optuna
import evaluate
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DefaultDataCollator
)
from peft import LoraConfig, get_peft_model
from huggingface_hub import HfApi
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

In [ ]:
# gets the peak memory VRAM, then resets for next iteration testing
def gpu_memory_gb():
    peak = torch.cuda.max_memory_allocated() / 1024**3
    torch.cuda.reset_peak_memory_stats()
    return round(peak, 3)

# Dataset

Sentiment140, 5k train, 1k test

https://huggingface.co/datasets/bdanko/sentiment140

In [2]:
# 5k train, 1k for testing
def prepare_dataset(dataset_name="bdanko/sentiment140", train_size=5000, test_size=1000):
    print(f"Loading dataset {dataset_name}...")
    dataset = load_dataset(dataset_name, split="train")
    df = dataset.to_pandas()

    # negative sentiment swapped to 1
    df['sentiment'] = df['sentiment'].replace(4, 1)

    train_df = df.sample(n=train_size, weights=df['sentiment'].map({0: 0.5, 1: 0.5}), random_state=SEED)
    remaining_df = df.drop(train_df.index)

    test_df_neg = remaining_df[remaining_df['sentiment'] == 0].sample(n=test_size // 2, random_state=SEED)
    test_df_pos = remaining_df[remaining_df['sentiment'] == 1].sample(n=test_size // 2, random_state=SEED)
    test_df = pd.concat([test_df_neg, test_df_pos]).sample(frac=1, random_state=SEED)

    return DatasetDict({
        "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
        "test": Dataset.from_pandas(test_df.reset_index(drop=True))
    })

In [5]:
dataset = prepare_dataset()
print(dataset)

Loading dataset bdanko/sentiment140...


README.md:   0%|          | 0.00/420 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/113M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1600000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'date', 'user', 'sentiment', 'query'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'date', 'user', 'sentiment', 'query'],
        num_rows: 1000
    })
})


In [6]:
model_id = "mistralai/Mistral-7B-v0.3"
# We stick with the base model,
# because we do not need to care about instruction alignment
# we are training a classifier head that we don't need to explicitly
# care about generation parsing

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [7]:
def tokenize_function(examples):
    texts = [f"Sentiment classification. Text: {text}\nSentiment: " for text in examples["text"]]
    tokenized = tokenizer(texts, truncation=True, max_length=128, padding="max_length")
    tokenized["labels"] = examples["sentiment"]
    return tokenized

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
def get_base_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id,
        num_labels=2,
        dtype=torch.bfloat16,
        device_map={"": 0}, # everything must be on one
                            # GPU and namespace or optuna complains
      )
    model.config.pad_token_id = model.config.eos_token_id
    return model

In [ ]:
base_model = get_base_model()

In [9]:
# huggingface standardized metrics for:
# accuracy
# precision
# recll
# f1

accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision_metric.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall_metric.compute(predictions=predictions, references=labels)["recall"],
        "f1": f1_metric.compute(predictions=predictions, references=labels)["f1"],
    }

In [11]:
base_model.eval()
all_logits = []
all_labels = []

for i in range(0, len(tokenized_datasets["test"]), 8):
    batch = tokenized_datasets["test"][i : i + 8]
    inputs = {k: torch.tensor(v).to(base_model.device) for k, v in batch.items() if k != 'labels'}
    with torch.no_grad():
        logits = base_model(**inputs).logits

        # numpy cannot compute bfloat16
        all_logits.append(logits.float().cpu().numpy())
        all_labels.extend(batch['labels'])

base_logits_concatenated = np.concatenate(all_logits, axis=0)
base_results_raw = compute_metrics((base_logits_concatenated, np.array(all_labels)))
base_results = {f"eval_{k}": v for k, v in base_results_raw.items()}

In [12]:
# the base results are random,
# as the classification head is not trained
base_results

{'eval_accuracy': 0.499,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0}

In [13]:
def lora_objective(trial):

    # testing ranks 4, 8, 16, 32 (checking for best rank)
    r     = trial.suggest_categorical("r", [4, 8, 16, 32])

    # testing for best alpha
    alpha = trial.suggest_categorical("alpha", [16, 32, 64])

    # testing for learning rates between 1e-5, 5e-4
    lr    = trial.suggest_float("lr", 1e-5, 5e-4, log=True)

    peft_config = LoraConfig(
        r=r, lora_alpha=alpha, target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1, bias="none", task_type="SEQ_CLS",
    )

    torch.cuda.reset_peak_memory_stats()
    trial_model = get_base_model()
    trial_model.add_adapter(peft_config, adapter_name="lora_trial")

    training_args = TrainingArguments(
        output_dir="./lora_results",
        per_device_train_batch_size=16,
        num_train_epochs=1,
        weight_decay=0.01,
        learning_rate=lr,
        eval_strategy="steps",
        eval_steps=50,
        logging_steps=50,
        bf16=True,
        report_to="none",
        save_strategy="no",
        load_best_model_at_end=False,
        disable_tqdm=True,
        #gradient_checkpointing=True,
        #gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    trainer = Trainer(
        model=trial_model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    trial.report(eval_results["eval_f1"], step=1)
    if trial.should_prune():
        del trainer, trial_model
        torch.cuda.empty_cache()
        raise optuna.exceptions.TrialPruned()

    trial.set_user_attr("peak_vram_gb", gpu_memory_gb())

    del trainer
    del trial_model
    import gc
    gc.collect() # Force Python to find unreferenced objects
    torch.cuda.empty_cache() # Clear the PyTorch cache
    torch.cuda.reset_peak_memory_stats()

    return eval_results["eval_f1"]

In [11]:
# the study crashes frequently and OOMs after each
# study run
# Save results in a db across sessions if we restart

db_path = "sqlite:///lora_study.db"

# LoRA Study Run

In [14]:
# we set for 10 optuna trials
# this is total, so it mixes and matches
# 30-50 would be best time permitting

lora_study = optuna.create_study(
    study_name="mistral_sentiment",
    storage=db_path,
    load_if_exists=True,
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1)
)

lora_study.optimize(lora_objective, n_trials=10)

[I 2026-04-25 01:26:42,971] Using an existing study with name 'mistral_sentiment' instead of creating a new one.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '0.7118', 'grad_norm': '110.5', 'learning_rate': '2.335e-05', 'epoch': '0.1597'}
{'eval_loss': '0.5028', 'eval_accuracy': '0.802', 'eval_precision': '0.7736', 'eval_recall': '0.854', 'eval_f1': '0.8118', 'eval_runtime': '11.53', 'eval_samples_per_second': '86.7', 'eval_steps_per_second': '10.84', 'epoch': '0.1597'}
{'loss': '0.5564', 'grad_norm': '66.5', 'learning_rate': '1.892e-05', 'epoch': '0.3195'}
{'eval_loss': '0.4832', 'eval_accuracy': '0.795', 'eval_precision': '0.9086', 'eval_recall': '0.656', 'eval_f1': '0.7619', 'eval_runtime': '11.45', 'eval_samples_per_second': '87.35', 'eval_steps_per_second': '10.92', 'epoch': '0.3195'}
{'loss': '0.3632', 'grad_norm': '39', 'learning_rate': '1.45e-05', 'epoch': '0.4792'}
{'eval_loss': '0.3686', 'eval_accuracy': '0.866', 'eval_precision': '0.872', 'eval_recall': '0.858', 'eval_f1': '0.8649', 'eval_runtime': '11.54', 'eval_samples_per_second': '86.62', 'eval_steps_per_second': '10.83', 'epoch': '0.4792'}
{'loss': '0.4359', 'grad_n

[I 2026-04-25 01:30:19,207] Trial 1 finished with value: 0.8685258964143426 and parameters: {'r': 4, 'alpha': 64, 'lr': 2.76785974603523e-05}. Best is trial 1 with value: 0.8685258964143426.


...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'loss': '1.151', 'grad_norm': '129', 'learning_rate': '0.0003594', 'epoch': '0.1597'}
{'eval_loss': '0.7177', 'eval_accuracy': '0.756', 'eval_precision': '0.8033', 'eval_recall': '0.678', 'eval_f1': '0.7354', 'eval_runtime': '11.53', 'eval_samples_per_second': '86.7', 'eval_steps_per_second': '10.84', 'epoch': '0.1597'}
{'loss': '0.6944', 'grad_norm': '17.25', 'learning_rate': '0.0002913', 'epoch': '0.3195'}
{'eval_loss': '0.7816', 'eval_accuracy': '0.862', 'eval_precision': '0.9022', 'eval_recall': '0.812', 'eval_f1': '0.8547', 'eval_runtime': '11.54', 'eval_samples_per_second': '86.65', 'eval_steps_per_second': '10.83', 'epoch': '0.3195'}
{'loss': '0.5275', 'grad_norm': '55.5', 'learning_rate': '0.0002232', 'epoch': '0.4792'}
{'eval_loss': '0.677', 'eval_accuracy': '0.782', 'eval_precision': '0.9147', 'eval_recall': '0.622', 'eval_f1': '0.7405', 'eval_runtime': '11.54', 'eval_samples_per_second': '86.67', 'eval_steps_per_second': '10.83', 'epoch': '0.4792'}
{'loss': '0.5149', 'grad_

[I 2026-04-25 02:02:37,040] Trial 10 pruned. 


{'eval_loss': '0.3259', 'eval_accuracy': '0.887', 'eval_precision': '0.8832', 'eval_recall': '0.892', 'eval_f1': '0.8876', 'eval_runtime': '11.52', 'eval_samples_per_second': '86.8', 'eval_steps_per_second': '10.85', 'epoch': '1'}


In [ ]:
import pandas as pd

def summarize_study(study_name, method_label):
    try:
        # Load the study from the database
        study = optuna.load_study(study_name=study_name, storage=db_path)

        # Convert to dataframe
        df = study.trials_dataframe()

        # Filter for completed trials and clean up column names
        df = df[df['state'] == 'COMPLETE'].copy()

        # Extract user attributes (Peak VRAM)
        df['peak_vram_gb'] = df['user_attrs_peak_vram_gb']

        print(f"\n### Summary for {method_label} ({study_name}) ###")
        print(f"Best Trial F1: {study.best_value:.4f}")
        print(f"Best Parameters: {study.best_params}")

        # Return a simplified view of all trials
        cols_to_show = [col for col in df.columns if col.startswith('params_') or col in ['value', 'peak_vram_gb']]
        return df[cols_to_show].sort_values(by='value', ascending=False)

    except Exception as e:
        return f"Could not load {study_name}: {e}"

# Summarize LoRA
lora_summary = summarize_study("mistral_sentiment", "LoRA")
display(lora_summary.head())

In [ ]:
def train_best(study, is_lora=True):
    best_params = study.best_params
    if is_lora:
        peft_config = LoraConfig(
            r=best_params['r'], lora_alpha=best_params['alpha'],
            target_modules=["q_proj", "v_proj"], lora_dropout=0.1,
            bias="none", task_type="SEQ_CLS",
        )
        lr      = best_params['lr']
        out_dir = "./best_lora"
        hub_id  = "bdanko/mistral-7b-sentiment-lora"
    else:
        peft_config = IA3Config(
            target_modules=["k_proj", "v_proj", "down_proj"],
            feedforward_modules=["down_proj"],
            task_type="SEQ_CLS",
        )
        lr      = best_params['lr']
        out_dir = "./best_adapter"
        hub_id  = "bdanko/mistral-7b-sentiment-adapter"

    torch.cuda.reset_peak_memory_stats()
    model = get_base_model()
    if not is_lora:
        dropout = best_params.get('dropout', 0.0)
        for layer in model.model.layers:
            layer.self_attn.dropout = dropout

    model.add_adapter(peft_config, adapter_name="best_adapter")

    args = TrainingArguments(
        output_dir=out_dir,
        per_device_train_batch_size=16,
        num_train_epochs=3,
        learning_rate=lr,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        bf16=True,
        report_to="none",
        push_to_hub=True,
        hub_model_id=hub_id,
        #gradient_checkpointing=True,
        #gradient_checkpointing_kwargs={"use_reentrant": False},
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        compute_metrics=compute_metrics,
    )

    trainer.train()
    final_metrics = trainer.evaluate()
    peak_vram = gpu_memory_gb()
    return trainer, final_metrics, peak_vram

In [ ]:
# we train the model that has gotten the best lora
best_lora_trainer, best_lora_metrics, lora_vram = train_best(lora_study, True)

In [ ]:
lora_trials_df    = lora_study.trials_dataframe()
lora_trials_df["method"]    = "LoRA"
all_trials = pd.concat([lora_trials_df, adapter_trials_df], ignore_index=True)
all_trials.to_csv("peft_sentiment_optuna_study.csv", index=False)
print("Study saved locally → peft_sentiment_optuna_study.csv")

hf_dataset = Dataset.from_pandas(all_trials)
hf_dataset.push_to_hub("bdanko/peft-sentiment-optuna-study")
print("Study pushed to HF Hub → bdanko/peft-sentiment-optuna-study")

print(f"\nPeak VRAM — LoRA final training : {lora_vram:.3f} GB")

lp = lora_study.best_params
ap = adapter_study.best_params

results = pd.DataFrame([
    {
        "Method":    "Base LLM (Zero-Shot)",
        "Best Hyperparams": "N/A",
        "Accuracy":  base_results.get('eval_accuracy', '—'),
        "Precision": base_results.get('eval_precision', '—'),
        "Recall":    base_results.get('eval_recall', '—'),
        "F1":        base_results.get('eval_f1', '—'),
        "Peak VRAM (GB)": '—',
    },
    {
        "Method":    "LoRA (FT)",
        "Best Hyperparams": f"r={lp['r']}, α={lp['alpha']}, lr={lp['lr']:.2e}",
        "Accuracy":  best_lora_metrics.get('eval_accuracy', '—'),
        "Precision": best_lora_metrics.get('eval_precision', '—'),
        "Recall":    best_lora_metrics.get('eval_recall', '—'),
        "F1":        best_lora_metrics.get('eval_f1', '—'),
        "Peak VRAM (GB)": lora_vram,
    },
])

print(results.to_string(index=False))

# Qualitative Analysis

In [ ]:
def get_errors(trainer, ds):
    preds       = trainer.predict(ds)
    predictions = np.argmax(preds.predictions, axis=-1)
    labels      = preds.label_ids
    errors      = np.where(predictions != labels)[0]
    return errors, predictions

lora_errors,    lora_preds    = get_errors(best_lora_trainer,    tokenized_datasets["test"])
adapter_errors, adapter_preds = get_errors(best_adapter_trainer, tokenized_datasets["test"])

raw_test = dataset["test"]
label_map = {0: "Negative", 1: "Positive"}

both_wrong = np.intersect1d(lora_errors, adapter_errors)
sample_ids = both_wrong[:3] if len(both_wrong) >= 3 else lora_errors[:3]

print("=" * 70)
print("ERROR ANALYSIS — 3 hard misclassification samples")
print("=" * 70)
for rank, idx in enumerate(sample_ids, 1):
    idx = int(idx)
    true_lbl    = label_map[int(raw_test[idx]['sentiment'])]
    lora_lbl    = label_map[int(lora_preds[idx])]
    adapter_lbl = label_map[int(adapter_preds[idx])]
    print(f"\nSample {rank} (index {idx}):")
    print(f"  Text    : {raw_test[idx]['text']}")
    print(f"  True    : {true_lbl}")
    print(f"  LoRA    : {lora_lbl}  {'✅' if lora_lbl == true_lbl else '❌'}")
    print(f"  IA³     : {adapter_lbl}  {'✅' if adapter_lbl == true_lbl else '❌'}")

print("\n" + "=" * 70)
print("TRAINING STABILITY & VRAM COMPARISON")
print("=" * 70)
lora_vram_trials    = [t.user_attrs.get('peak_vram_gb', float('nan')) for t in lora_study.trials]
adapter_vram_trials = [t.user_attrs.get('peak_vram_gb', float('nan')) for t in adapter_study.trials]
print(f"LoRA    — avg trial VRAM: {np.nanmean(lora_vram_trials):.3f} GB  "
      f"| final training VRAM: {lora_vram:.3f} GB")
print(f"IA³     — avg trial VRAM: {np.nanmean(adapter_vram_trials):.3f} GB  "
      f"| final training VRAM: {adapter_vram:.3f} GB")
lora_f1s    = [t.value for t in lora_study.trials if t.value is not None]
adapter_f1s = [t.value for t in adapter_study.trials if t.value is not None]
print(f"LoRA    — F1 std across 20 trials: {np.std(lora_f1s):.4f}  (lower = more stable)")
print(f"IA³     — F1 std across 10 trials: {np.std(adapter_f1s):.4f}")

# Report

Confusion matrices, learning curve plots

In [ ]:
def plot_confusion_matrix(trainer, ds, title, filename):
    preds = trainer.predict(ds)
    predictions = np.argmax(preds.predictions, axis=-1)
    labels = preds.label_ids
    cm = confusion_matrix(labels, predictions)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Neg", "Pos"], yticklabels=["Neg", "Pos"])
    plt.title(f"Confusion Matrix: {title}")
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.savefig(filename)
    plt.show()

print("\nGenerating confusion matrices...")
plot_confusion_matrix(best_lora_trainer, tokenized_datasets["test"], "Best LoRA", "lora_cm.png")
plot_confusion_matrix(best_adapter_trainer, tokenized_datasets["test"], "Best IA³", "adapter_cm.png")

def plot_learning_curves(trainer, title, filename):
    history = trainer.state.log_history
    steps = [h['step'] for h in history if 'loss' in h]
    loss = [h['loss'] for h in history if 'loss' in h]
    eval_steps = [h['step'] for h in history if 'eval_loss' in h]
    eval_loss = [h['eval_loss'] for h in history if 'eval_loss' in h]

    plt.figure(figsize=(8, 5))
    plt.plot(steps, loss, label="Train Loss")
    plt.plot(eval_steps, eval_loss, label="Eval Loss", linestyle='--')
    plt.title(f"Learning Curve: {title}")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.legend()
    plt.savefig(filename)
    plt.show()

print("Generating learning curves...")
plot_learning_curves(best_lora_trainer, "Best LoRA", "lora_loss.png")
plot_learning_curves(best_adapter_trainer, "Best IA³", "adapter_loss.png")(base)